# LP Step 3 — Silver Layer
### Clean, Type-Cast & Join Across SAP + Salesforce + TMS

**Output tables:**
- `silver_orders` — SAP orders joined with invoices
- `silver_accounts` — Salesforce accounts joined with best contract
- `silver_shipments` — TMS shipments with delay categories
- `silver_unified_customer_view` — Customer 360 (3-way join)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

# Source Bronze tables
BRONZE_SAP_ORDERS    = "supply_chain_dev.lp_analytics.bronze_sap_orders"
BRONZE_SAP_INVOICES  = "supply_chain_dev.lp_analytics.bronze_sap_invoices"
BRONZE_SF_ACCOUNTS   = "supply_chain_dev.lp_analytics.bronze_sf_accounts"
BRONZE_SF_CONTRACTS  = "supply_chain_dev.lp_analytics.bronze_sf_contracts"
BRONZE_TMS_SHIPMENTS = "supply_chain_dev.lp_analytics.bronze_tms_shipments"

# Silver target tables
SILVER_ORDERS     = "supply_chain_dev.lp_analytics.silver_orders"
SILVER_ACCOUNTS   = "supply_chain_dev.lp_analytics.silver_accounts"
SILVER_SHIPMENTS  = "supply_chain_dev.lp_analytics.silver_shipments"
SILVER_UNIFIED    = "supply_chain_dev.lp_analytics.silver_unified_customer_view"

print("Config ready!")
print(f"Sources : 5 Bronze tables")
print(f"Targets : {SILVER_ORDERS}")
print(f"          {SILVER_ACCOUNTS}")
print(f"          {SILVER_SHIPMENTS}")
print(f"          {SILVER_UNIFIED}")

Config ready!
Sources : 5 Bronze tables
Targets : supply_chain_dev.lp_analytics.silver_orders
          supply_chain_dev.lp_analytics.silver_accounts
          supply_chain_dev.lp_analytics.silver_shipments
          supply_chain_dev.lp_analytics.silver_unified_customer_view


## Step 3.1 — Silver Orders
SAP orders LEFT JOINed with invoices on `sap_order_id`.
Result: 300 records with revenue, margin, and payment status per order.

In [0]:
# Read and clean SAP Orders
sap_orders_clean = spark.table(BRONZE_SAP_ORDERS) \
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd")) \
    .withColumn("extract_date", to_date(col("extract_date"), "yyyy-MM-dd")) \
    .filter(col("sap_order_id").isNotNull()) \
    .filter(col("customer_id").isNotNull()) \
    .filter(col("revenue_usd") > 0) \
    .drop("source_file", "ingested_at")

# Read and clean SAP Invoices
sap_invoices_clean = spark.table(BRONZE_SAP_INVOICES) \
    .withColumn("invoice_date", to_date(col("invoice_date"), "yyyy-MM-dd")) \
    .filter(col("invoice_id").isNotNull()) \
    .select(
        "sap_order_id", "invoice_id", "invoice_date",
        "invoice_amount_usd", "payment_status", "days_to_pay"
    )

# LEFT JOIN orders with invoices on sap_order_id
# Keeps all 300 orders even if 50 don't have invoices yet
silver_orders_df = sap_orders_clean.alias("o") \
    .join(sap_invoices_clean.alias("i"),
          col("o.sap_order_id") == col("i.sap_order_id"), "left") \
    .select(
        col("o.sap_order_id"), col("o.customer_id"),
        col("o.customer_name"), col("o.order_date"),
        col("o.service_type"), col("o.trade_lane"),
        col("o.incoterms"), col("o.freight_cost_usd"),
        col("o.revenue_usd"), col("o.margin_usd"),
        col("o.margin_pct"), col("o.weight_kg"),
        col("o.volume_cbm"), col("o.sap_division"),
        col("o.payment_terms"), col("o.order_status"),
        col("o.created_by"), col("i.invoice_id"),
        col("i.invoice_date"), col("i.invoice_amount_usd"),
        col("i.payment_status"), col("i.days_to_pay"),
        current_timestamp().alias("silver_loaded_at")
    )

silver_orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(SILVER_ORDERS)

print(f"Silver Orders: {spark.table(SILVER_ORDERS).count()} records")
display(spark.table(SILVER_ORDERS).select(
    "sap_order_id", "customer_id", "service_type",
    "revenue_usd", "margin_pct", "payment_status"
).limit(5))

## Step 3.2 — Silver Accounts
Salesforce accounts joined with highest-value contract per customer using window function.
Result: 15 records with NPS, account status, and contract value.

In [0]:
# Clean Salesforce Accounts
sf_accounts_clean = spark.table(BRONZE_SF_ACCOUNTS) \
    .withColumn("created_date", to_date(col("created_date"), "yyyy-MM-dd")) \
    .withColumn("last_activity_date", to_date(col("last_activity_date"), "yyyy-MM-dd")) \
    .filter(col("customer_id").isNotNull()) \
    .drop("source_file", "ingested_at")

# Clean Salesforce Contracts — get best contract per customer
# (some customers have multiple contracts — take highest value active one)
from pyspark.sql.window import Window

contract_window = Window.partitionBy("customer_id").orderBy(col("contract_value_usd").desc())

sf_contracts_clean = spark.table(BRONZE_SF_CONTRACTS) \
    .withColumn("start_date", to_date(col("start_date"), "yyyy-MM-dd")) \
    .withColumn("end_date", to_date(col("end_date"), "yyyy-MM-dd")) \
    .withColumn("rank", row_number().over(contract_window)) \
    .filter(col("rank") == 1) \
    .drop("rank", "source_file", "ingested_at") \
    .select(
        "customer_id", "contract_id", "contract_type",
        "contract_value_usd", "contract_status",
        "renewal_probability_pct", "assigned_rep",
        "start_date", "end_date"
    )

# Join accounts with their best contract
silver_accounts_df = sf_accounts_clean.alias("a") \
    .join(sf_contracts_clean.alias("c"),
          col("a.customer_id") == col("c.customer_id"),
          "left") \
    .select(
        col("a.customer_id"), col("a.sf_account_id"),
        col("a.account_name"), col("a.segment"),
        col("a.region"), col("a.account_owner"),
        col("a.industry"), col("a.annual_revenue_usd"),
        col("a.employee_count"), col("a.account_status"),
        col("a.nps_score"), col("a.created_date"),
        col("a.last_activity_date"),
        col("c.contract_id"), col("c.contract_type"),
        col("c.contract_value_usd"), col("c.contract_status"),
        col("c.renewal_probability_pct"), col("c.assigned_rep"),
        current_timestamp().alias("silver_loaded_at")
    )

silver_accounts_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(SILVER_ACCOUNTS)

print(f"Silver Accounts: {spark.table(SILVER_ACCOUNTS).count()} records")
display(spark.table(SILVER_ACCOUNTS).select(
    "customer_id", "account_name", "segment",
    "account_status", "nps_score", "contract_value_usd",
    "renewal_probability_pct"
).limit(5))


Silver Accounts: 15 records


customer_id,account_name,segment,account_status,nps_score,contract_value_usd,renewal_probability_pct
CUST001,Amazon Logistics,Enterprise,At Risk,50,345886.9,46
CUST002,Walmart Supply Chain,Enterprise,Active,72,4037350.61,32
CUST003,Target Distribution,Enterprise,At Risk,77,3675391.93,38
CUST004,Home Depot Freight,Mid-Market,Active,26,2297690.95,53
CUST005,Nike Global Logistics,Enterprise,At Risk,21,3618967.64,93


## Step 3.3 — Silver Shipments
TMS shipments cleaned with delay category labels.
Result: 400 records with On Time / Minor / Moderate / Severe Delay classification.

In [0]:
silver_shipments_df = spark.table(BRONZE_TMS_SHIPMENTS) \
    .withColumn("ship_date", to_date(col("ship_date"), "yyyy-MM-dd")) \
    .withColumn("eta", to_date(col("eta"), "yyyy-MM-dd")) \
    .withColumn("actual_arrival", to_date(col("actual_arrival"), "yyyy-MM-dd")) \
    .filter(col("shipment_id").isNotNull()) \
    .filter(col("customer_id").isNotNull()) \
    .withColumn("on_time_delivery", col("on_time_delivery").cast(BooleanType())) \
    .withColumn("is_delayed", when(col("delay_days") > 0, True).otherwise(False)) \
    .withColumn("delay_category",
        when(col("delay_days") == 0, "On Time")
        .when(col("delay_days") <= 3, "Minor Delay")
        .when(col("delay_days") <= 7, "Moderate Delay")
        .otherwise("Severe Delay")
    ) \
    .drop("source_file", "ingested_at") \
    .withColumn("silver_loaded_at", current_timestamp())

silver_shipments_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(SILVER_SHIPMENTS)

print(f"Silver Shipments: {spark.table(SILVER_SHIPMENTS).count()} records")
display(spark.table(SILVER_SHIPMENTS).select(
    "shipment_id", "customer_id", "carrier",
    "service_type", "delay_days", "delay_category",
    "on_time_delivery"
).limit(5))

Silver Shipments: 400 records


shipment_id,customer_id,carrier,service_type,delay_days,delay_category,on_time_delivery
SHP000001,CUST001,CMA CGM,FCL,0,On Time,true
SHP000002,CUST010,CMA CGM,FCL,0,On Time,true
SHP000003,CUST008,FedEx Freight,Cross Border Truck,0,On Time,true
SHP000004,CUST015,DHL Global,FCL,0,On Time,true
SHP000005,CUST007,COSCO,Air Freight,0,On Time,true


## Step 3.4 — Silver Unified Customer View (Customer 360)
**This is the most important table.** 3-way join across all three systems on `customer_id`.
Result: 15 rows — one per customer — showing revenue + NPS + on-time delivery in one place.

In [0]:
# Aggregate SAP orders per customer
sap_agg = spark.table(SILVER_ORDERS) \
    .groupBy("customer_id") \
    .agg(
        count("sap_order_id").alias("total_orders"),
        round(sum("revenue_usd"), 2).alias("total_revenue_usd"),
        round(sum("margin_usd"), 2).alias("total_margin_usd"),
        round(avg("margin_pct"), 2).alias("avg_margin_pct"),
        round(sum("freight_cost_usd"), 2).alias("total_freight_cost_usd"),
        countDistinct("service_type").alias("service_types_used"),
        sum(when(col("payment_status") == "OVERDUE", 1).otherwise(0)).alias("overdue_invoices"),
        sum(when(col("payment_status") == "PAID", 1).otherwise(0)).alias("paid_invoices")
    )

# Aggregate TMS shipments per customer
tms_agg = spark.table(SILVER_SHIPMENTS) \
    .groupBy("customer_id") \
    .agg(
        count("shipment_id").alias("total_shipments"),
        round(avg(col("on_time_delivery").cast("int")) * 100, 1).alias("on_time_pct"),
        round(avg("delay_days"), 1).alias("avg_delay_days"),
        sum(when(col("delay_category") == "Severe Delay", 1).otherwise(0)).alias("severe_delays"),
        countDistinct("carrier").alias("carriers_used"),
        round(sum("freight_charges_usd"), 2).alias("total_freight_charges_usd")
    )

# Join all three: Salesforce + SAP + TMS
unified_df = spark.table(SILVER_ACCOUNTS).alias("sf") \
    .join(sap_agg.alias("sap"),
          col("sf.customer_id") == col("sap.customer_id"), "left") \
    .join(tms_agg.alias("tms"),
          col("sf.customer_id") == col("tms.customer_id"), "left") \
    .select(
        col("sf.customer_id"),
        col("sf.account_name"),
        col("sf.segment"),
        col("sf.region"),
        col("sf.account_status"),
        col("sf.nps_score"),
        col("sf.contract_value_usd"),
        col("sf.renewal_probability_pct"),
        col("sf.assigned_rep"),
        col("sap.total_orders"),
        col("sap.total_revenue_usd"),
        col("sap.total_margin_usd"),
        col("sap.avg_margin_pct"),
        col("sap.total_freight_cost_usd"),
        col("sap.overdue_invoices"),
        col("sap.paid_invoices"),
        col("tms.total_shipments"),
        col("tms.on_time_pct"),
        col("tms.avg_delay_days"),
        col("tms.severe_delays"),
        col("tms.carriers_used"),
        current_timestamp().alias("silver_loaded_at")
    )

unified_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(SILVER_UNIFIED)

print(f"Unified Customer View: {spark.table(SILVER_UNIFIED).count()} customers")
display(spark.table(SILVER_UNIFIED).select(
    "customer_id", "account_name", "segment",
    "total_orders", "total_revenue_usd",
    "on_time_pct", "overdue_invoices", "nps_score"
).orderBy(desc("total_revenue_usd")))

Unified Customer View: 15 customers


customer_id,account_name,segment,total_orders,total_revenue_usd,on_time_pct,overdue_invoices,nps_score
CUST001,Amazon Logistics,Enterprise,26,265636.0,83.9,9,50
CUST003,Target Distribution,Enterprise,18,203850.42,69.4,3,77
CUST013,Procter & Gamble,Enterprise,20,195103.26,80.0,7,21
CUST004,Home Depot Freight,Mid-Market,20,194681.54,70.4,8,26
CUST010,Best Buy Logistics,Mid-Market,23,182697.54,81.5,7,86
CUST007,Samsung Americas,Mid-Market,26,178989.35,74.2,9,30
CUST011,Zara Supply Chain,Mid-Market,18,178119.08,84.6,4,40
CUST014,Unilever Logistics,Mid-Market,21,170759.1,83.3,7,25
CUST002,Walmart Supply Chain,Enterprise,24,168664.08,64.3,6,72
CUST012,Tesla Parts Logistics,Enterprise,18,159033.65,66.7,5,45
